Imports

In [3]:
import torch
from torch import nn
from torch import optim
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms

Creating a FC Network

In [4]:
class NN(nn.Module):
    def __init__(self, inputSize, numClasses):
        super(NN, self).__init__()
        self.fc1 = nn.Linear(inputSize, 50)
        self.fc2 = nn.Linear(50, numClasses)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

Setting device to GPU

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
INPUT_SIZE = 784
NUM_CLASSES = 10
BATCH_SIZE = 64
EPOCHS = 1
LEARNING_RATE = .001

Loading dataset tensors

In [7]:
trainDataset = datasets.MNIST(root='dataset/', train=True, transform=transforms.ToTensor(), download=True)
trainLoader = DataLoader(dataset=trainDataset, batch_size=BATCH_SIZE, shuffle=True)

testDataset = datasets.MNIST(root='dataset/', train=False, transform=transforms.ToTensor(), download=True)
testLoader = DataLoader(dataset=testDataset, batch_size=BATCH_SIZE, shuffle=True)

100.0%
100.0%
100.0%
100.0%


Creating a model

In [8]:
model = NN(INPUT_SIZE, NUM_CLASSES).to(device)

Loss & Optimizer

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

Training the network

In [ ]:
for epoch in range(EPOCHS):
    for batch_idx, (data, targets) in enumerate(trainLoader):
        data = data.to(device)
        targets = targets.to(device)
        # shifting the tensor to GPU will make every tensor calc on GPU itself, making it much faster than doing it in CPU

        # data.shape = torch.Size([64, 1, 28, 28]) => 64 is batch size (each batch containes 64 training samples), 1 is single channel color - grayscale, (28, 28) = image resolution 28x28 pixels
        # so now we flatten 28x28 to 784
        data = data.reshape(data.shape[0], -1)

        # Forward pass
        scores = model(data)
        loss = criterion(scores, targets)

        # backward pass
        optimizer.zero_grad()
        loss.backward()

        # Adam step
        optimizer.step() # Here we update the weights

Checking Accuracy on training & test dataset

In [15]:
def checkAccuracy(loader, model):
    if loader.dataset.train:print("Checking accuracy on Training dataset")
    else : print("checking accuracy on testing dataset")
    num_correct = 0
    num_samples = 0
    model.eval() # This is switching model to evaluate to let it know that whatever changes may happen not to update it because the model isnt getting trained

    with torch.no_grad(): # we do this to stop calculating gradients and updating it
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            x = x.reshape(x.shape[0], -1)
            
            scores = model(x)
            _, predictions = scores.max(1)
            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)

        print(f"Got {num_correct} out of {num_samples} with accuracy {float(num_correct)/float(num_samples)*100:.2f}%")

    model.train() # switching back the model to training mode, but in this case as we have already trained the model this isn't necessary

In [16]:
checkAccuracy(trainLoader, model)
checkAccuracy(testLoader, model)

Checking accuracy on Training dataset
Got 56008 out of 60000 with accuracy 93.35%
checking accuracy on testing dataset
Got 9317 out of 10000 with accuracy 93.17%
